# C1.1 · The agentic offensive workflow, and containing it

**Function C — Red Teaming and Security Research with AI → Red Teaming with AI**  ·  *Both directions*

Builds on **[C1.0 · Start here — what red teaming and research with AI means](https://spbreed.github.io/cyber-commons/lessons/C1.0.html)**.

| | |
|---|---|
| Open-source tooling | CAI, Metasploit, Firecracker |
| Open-weight models | Kimi K2, GLM-4.6 |
| Frontier models | Claude Sonnet 5 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

An offensive harness reads only hostile input, by definition: every byte comes from a system you are attacking. It is the most dangerous agent in the building, and the thing that makes running it professional is that scope stops living in the tester's attention.

## 2 · The framework

```
   recon --> hypothesis --> test --> escalate --> report
        (the loop has not changed; who runs each turn has)

   +--------------------------------------------------+
   |  harness scope check   host in engagement set?   |
   +--------------------------------------------------+
   |  sandbox egress        private/link-local? rate? |
   +--------------------------------------------------+
              two layers, neither of them the model

   everything the harness reads is hostile by design
```

Penetration testing has always been a loop: **recon → hypothesis → test →
escalate → report.** What has changed is who runs each turn.

**Manual (still the baseline).** A human runs `nmap`, reads the output, forms a
hypothesis, tries it. Slow, and the quality is entirely the tester's.

**Scripted.** The recon is automated — Nuclei templates, a Burp scan. The
hypothesis and the escalation are still human. This is where most teams are.

**Semi-autonomous.** An open-weight model reads the recon output and *proposes*
which findings are worth chasing and what to try next. The human approves each
action. The gain is triage speed on a large surface: 400 findings ranked in
minutes rather than a day.

**Autonomous.** The model proposes and the harness executes, within a
pre-approved scope and tool set, verifying its own results. This is real and it
works, and it is also where the engagement becomes a safety problem — because an
agent that has not understood the scope will happily test something outside it
at machine speed.

The professional obligations do not change with autonomy. They get harder,
because scope enforcement can no longer live in the tester's attention — it has
to live in the harness, and then underneath the harness in the network.

That second half is why containment belongs in this lesson rather than in a
later one. An offensive harness has a property no other agent has: **everything
it reads is hostile by design.** Banner strings, error bodies, file contents —
all of it comes from a system you are attacking, which may itself already be
attacker-controlled. Containment there protects three parties at once: the
client (scope and rate limits, so you do not break their production), everyone
else (egress control, so a compromised harness cannot pivot outward), and you
(findings and client data must not leave by a route the agent chooses).

## 3 · The model backend, and the triage it proposes

In [ ]:
# --- model backend: replay by default, real model when you configure one ----
# Nothing here is Anthropic- or vendor-specific beyond one URL and one header
# shape. Standard library only, so the notebook stays self-contained.
import json, os, urllib.error, urllib.request

# The cheapest current model on each side, which is what a lesson needs.
FRONTIER_DEFAULT   = "claude-haiku-4-5-20251001"
OPEN_WEIGHT_DEFAULT = "glm-4.6"
TIMEOUT = 60

def backend():
    """(kind, model). Configuration comes from the environment, never a literal."""
    if os.environ.get("ANTHROPIC_API_KEY"):
        return "frontier", os.environ.get("MODEL", FRONTIER_DEFAULT)
    if os.environ.get("OPENAI_BASE_URL"):
        return "open-weight", os.environ.get("MODEL", OPEN_WEIGHT_DEFAULT)
    return "replay", "deterministic stand-in (no backend configured)"

def _post(url, payload, headers):
    req = urllib.request.Request(url, data=json.dumps(payload).encode(),
                                 headers={"content-type": "application/json", **headers})
    with urllib.request.urlopen(req, timeout=TIMEOUT) as r:
        return json.loads(r.read().decode())

def _anthropic(prompt, system, model, max_tokens, temperature):
    body = {"model": model, "max_tokens": max_tokens, "temperature": temperature,
            "messages": [{"role": "user", "content": prompt}]}
    if system:
        body["system"] = system
    out = _post("https://api.anthropic.com/v1/messages", body,
                {"x-api-key": os.environ["ANTHROPIC_API_KEY"],
                 "anthropic-version": "2023-06-01"})
    return "".join(b.get("text", "") for b in out.get("content", [])).strip()

def _openai_compatible(prompt, system, model, max_tokens, temperature):
    msgs = ([{"role": "system", "content": system}] if system else []) + \
           [{"role": "user", "content": prompt}]
    base = os.environ["OPENAI_BASE_URL"].rstrip("/")
    key = os.environ.get("OPENAI_API_KEY", "not-needed")
    out = _post(f"{base}/chat/completions",
                {"model": model, "messages": msgs, "max_tokens": max_tokens,
                 "temperature": temperature},
                {"authorization": f"Bearer {key}"})
    return out["choices"][0]["message"]["content"].strip()

def ask(prompt, *, replay, system=None, max_tokens=512, temperature=0.0):
    """Answer `prompt` with the configured backend, or return `replay`.

    `replay` is required, not optional: a lesson must be able to run offline,
    and the answer it falls back to has to be visible in the source rather than
    invented at runtime.
    """
    kind, model = backend()
    if kind == "replay":
        return replay, kind, model
    try:
        fn = _anthropic if kind == "frontier" else _openai_compatible
        return fn(prompt, system, model, max_tokens, temperature), kind, model
    except (urllib.error.URLError, urllib.error.HTTPError, KeyError, TimeoutError) as e:
        detail = getattr(e, "code", None) or type(e).__name__
        print(f"   !! {kind} backend ({model}) failed: {detail} - using the replay,")
        print("      which is a replay and is labelled as one. No model answered.")
        return replay, "replay", f"{model} unreachable"

_kind, _model = backend()
print(f"model backend : {_kind}")
print(f"model         : {_model}")
if _kind == "replay":
    print()
    print("This lesson runs offline against a deterministic replay, which is why")
    print("it works on a Kaggle kernel with the internet switched off. To run the")
    print("identical code against a real model, set one of:")
    print()
    print("   frontier     export ANTHROPIC_API_KEY=...   # cheapest: " + FRONTIER_DEFAULT)
    print("   open weight  export OPENAI_BASE_URL=http://localhost:11434/v1 \\")
    print("                       OPENAI_API_KEY=ollama MODEL=glm-4.6")

## 4 · The same lesson, against a real model

Everything below this point runs identically on three backends. Offline it uses
a deterministic replay that is labelled as a replay wherever it appears — never
presented as a model's output. With `ANTHROPIC_API_KEY` set it calls a frontier
model; with `OPENAI_BASE_URL` set it calls any OpenAI-compatible endpoint,
which covers Ollama, vLLM and the hosted open-weight providers.

The point of running it both ways is not that the answers match. It is that
**the lesson's assertion holds either way** — if it only holds against the
replay, the lesson was testing the replay.

In [ ]:
TASK = 'Rank these findings by which to chase first on an authorised engagement, and say why in one clause each.\nF-01 TLS 1.0 enabled on api.target.example\nF-02 /v1/users returns data without auth on api.target.example\nF-06 expired certificate on cdn.partner.example (not in scope)'

REPLAY = '1. F-02 - unauthenticated data endpoint, directly exploitable.\n2. F-01 - needs a downgrade position; no evidence of one here.\n3. F-06 - out of scope, do not touch.'

answer, used, model = ask(TASK, replay=REPLAY,
            system='You triage penetration-test findings. Ranked list, one clause each.',
            max_tokens=300)

print(f"backend used : {used}")
print(f"model        : {model}")
print(f"prompt       : {TASK[:66]}...")
print()
print("answer:")
for line in (answer.splitlines() or [answer]):
    print(f"   {line}")

# Two assertions that must hold on every backend, and one property that is
# reported rather than asserted - a real model failing it is a finding about
# the model, not a broken notebook.
assert answer.strip(), "the configured backend returned nothing"
if used == "replay":
    assert answer == REPLAY, "the offline path must return the replay verbatim"

label, held = ("put the unauthenticated endpoint first", answer.find("F-02") in range(0, 40))
print()
print(f"property checked : {label}")
print(f"held on {used:12s} : {held}")
print()
print("Same code, same assertions, three possible backends. Offline the answer")
print("is the replay and is labelled as one; with a key it is the model's.")

## 5 · Demo — the four generations on the same recon output

Realistic scan output from an authorised engagement against hosts you own. The question at every generation is the same: what do I chase first?

In [ ]:
FINDINGS = [
 {"id": "F-01", "host": "api.target.example",   "port": 443, "svc": "https",
  "note": "TLS 1.0 enabled",                       "sev": "medium", "exploitable": False},
 {"id": "F-02", "host": "api.target.example",   "port": 443, "svc": "https",
  "note": "/v1/users returns data without auth",   "sev": "high",   "exploitable": True},
 {"id": "F-03", "host": "www.target.example",   "port": 80,  "svc": "http",
  "note": "server banner discloses version",       "sev": "low",    "exploitable": False},
 {"id": "F-04", "host": "api.target.example",   "port": 22,  "svc": "ssh",
  "note": "password auth permitted",               "sev": "medium", "exploitable": True},
 {"id": "F-05", "host": "legacy.target.example", "port": 8080, "svc": "http",
  "note": "directory listing enabled on /backup",  "sev": "medium", "exploitable": True},
 {"id": "F-06", "host": "cdn.partner.example",  "port": 443, "svc": "https",
  "note": "expired certificate",                   "sev": "low",    "exploitable": False},
]
SEV_RANK = {"low": 1, "medium": 2, "high": 3, "critical": 4}

print("=== generation 1: manual — a human reads all six and decides ===")
print(f"   {len(FINDINGS)} findings, no ordering, ~20 min of reading\n")

print("=== generation 2: scripted — sort by severity ===")
for f in sorted(FINDINGS, key=lambda x: -SEV_RANK[x["sev"]])[:3]:
    print(f"   {f['id']} {f['sev']:7s} {f['note']}")
print("   → severity is a label, not a prediction. F-04 and F-05 are both")
print("     'medium' and both actually exploitable; F-01 is not.")

In [ ]:
# === generation 3: semi-autonomous — the model proposes, the human approves ===
class ReplayModel:
    """DETERMINISTIC REPLAY — not a language model. See the note above."""
    ASSESSMENT = {
     "F-01": (0.10, "TLS 1.0 needs a downgrade position; no evidence of one here"),
     "F-02": (0.95, "unauthenticated data endpoint — directly exploitable, chase first"),
     "F-03": (0.05, "banner disclosure alone is not a finding worth engagement time"),
     "F-04": (0.60, "password auth permits spraying if no lockout; test lockout first"),
     "F-05": (0.75, "directory listing on /backup often exposes archives with secrets"),
     "F-06": (0.02, "expired cert on a partner CDN — out of scope, do not touch"),
    }
    def triage(self, f):
        conf, why = self.ASSESSMENT[f["id"]]
        return {"id": f["id"], "priority": conf, "reasoning": why}

model = ReplayModel()
ranked = sorted((model.triage(f) for f in FINDINGS), key=lambda r: -r["priority"])
print("=== generation 3: semi-autonomous triage ===")
for r in ranked:
    print(f"   {r['id']}  p={r['priority']:.2f}  {r['reasoning']}")

truth = {f["id"]: f["exploitable"] for f in FINDINGS}
top3 = [r["id"] for r in ranked[:3]]
print(f"\n   top 3 chosen: {top3}")
print(f"   of which actually exploitable: "
      f"{sum(truth[i] for i in top3)}/3")
sev_top3 = [f["id"] for f in sorted(FINDINGS, key=lambda x: -SEV_RANK[x['sev']])[:3]]
print(f"   severity-sorted top 3: {sev_top3} → "
      f"{sum(truth[i] for i in sev_top3)}/3 exploitable")

## 6 · Where it breaks — generation 4, and the scope problem

The model's top-ranked item is correct. Its reasoning on F-06 is also correct — *out of scope, do not touch*. Now make it autonomous and remove the human from the loop. What stops it acting on a finding it has correctly identified as out of scope?

Nothing in the model. Its judgement about scope is a *proposal*, on the decision plane, exactly like everything else it produces.

In [ ]:
ENGAGEMENT_SCOPE = {"api.target.example", "www.target.example",
                    "legacy.target.example"}

def autonomous_no_enforcement(findings, model):
    """The model's own scope judgement is the only control. This is the bug."""
    acted = []
    for f in findings:
        r = model.triage(f)
        if r["priority"] > 0.5:
            acted.append((f["id"], f["host"]))
    return acted

def autonomous_enforced(findings, model, scope):
    """Scope is enforced by the harness, not believed from the model."""
    acted, refused = [], []
    for f in findings:
        if f["host"] not in scope:
            refused.append((f["id"], f["host"], "host outside the engagement scope"))
            continue
        r = model.triage(f)
        if r["priority"] > 0.5:
            acted.append((f["id"], f["host"]))
    return acted, refused

acted = autonomous_no_enforcement(FINDINGS, model)
print("no enforcement — acted on:", acted)

acted2, refused = autonomous_enforced(FINDINGS, model, ENGAGEMENT_SCOPE)
print("\nenforced — acted on:", acted2)
for fid, host, why in refused:
    print(f"   REFUSED {fid} ({host}): {why}")

print("\nBoth runs happen to avoid F-06 here, because the model ranked it 0.02.")
print("The difference is that one of them would still avoid it if the model")
print("ranked it 0.99. That is the whole distinction between a judgement and a control.")

In [ ]:
# Verify: fuzz the model's scope judgement. The control must hold regardless.
import random
random.seed(4)

class AdversarialModel(ReplayModel):
    """A model that has been convinced F-06 is critical — by a prompt injection,
    a bad fine-tune, or simply by being wrong."""
    def triage(self, f):
        if f["id"] == "F-06":
            return {"id": "F-06", "priority": 0.99, "reasoning": "critical, chase now"}
        return super().triage(f)

bad = AdversarialModel()
acted_unsafe = autonomous_no_enforcement(FINDINGS, bad)
acted_safe, refused_safe = autonomous_enforced(FINDINGS, bad, ENGAGEMENT_SCOPE)

out_of_scope_unsafe = [i for i, h in acted_unsafe if h not in ENGAGEMENT_SCOPE]
out_of_scope_safe   = [i for i, h in acted_safe   if h not in ENGAGEMENT_SCOPE]
print(f"model convinced F-06 is critical:")
print(f"   unenforced → out-of-scope actions: {out_of_scope_unsafe}")
print(f"   enforced   → out-of-scope actions: {out_of_scope_safe}")
assert not out_of_scope_safe
print("\nScope enforcement in the harness is what makes autonomy professionally")
print("defensible. Without it, your engagement letter is protected by a prompt.")

## 7 · The control — and the layer underneath it

The scope check above lives in the harness, which is one process away from the loop it constrains. On an engagement a single control is a single point of failure, and the failure is a professional incident. The same rule therefore gets restated where the agent cannot reach it: the sandbox's own request path.

In [ ]:
import re
from urllib.parse import urlparse
from dataclasses import dataclass, field

PRIVATE = [re.compile(p) for p in (r"^127\.", r"^10\.", r"^169\.254\.",
                                   r"^192\.168\.", r"^localhost$")]

@dataclass
class OffensiveSandbox:
    """Egress for the offensive harness. Refuses before the request is made."""
    scope: set
    rate_per_min: int = 60
    calls: list = field(default_factory=list)

    def request(self, url, at_minute=0):
        host = (urlparse(url).hostname or "").lower()
        if any(p.match(host) for p in PRIVATE):
            return False, "private/link-local address - not part of any engagement"
        if host not in self.scope:
            return False, f"host {host!r} is outside the engagement scope"
        if len([c for c in self.calls if c == at_minute]) >= self.rate_per_min:
            return False, (f"rate limit {self.rate_per_min}/min reached - "
                           f"protecting the client's production service")
        self.calls.append(at_minute)
        return True, "in scope, within rate"

box = OffensiveSandbox(scope=ENGAGEMENT_SCOPE, rate_per_min=2)
for url in ["https://api.target.example/v1/users",
            "https://api.target.example/v1/orders",
            "https://api.target.example/v1/admin",
            "https://cdn.partner.example/asset.js",
            "http://169.254.169.254/latest/meta-data/"]:
    ok, why = box.request(url)
    print(f"{'ALLOW' if ok else 'DENY ':5s} {url[:44]:46s} {why}")

print()
print("Three refusals for three different reasons: the client's rate limit, the")
print("engagement boundary, and the cloud metadata endpoint that is in nobody's")
print("scope. None of them consulted the model.")
assert not box.request("https://cdn.partner.example/x")[0]
assert not box.request("http://169.254.169.254/")[0]

## What you just proved

Severity sorting puts 2 of 3 exploitable findings in the top 3; model triage puts 3 of 3, and correctly reasons that the partner CDN is out of scope. With the model adversarially convinced that the out-of-scope host is critical, the unenforced harness acts on it and the enforced harness refuses. Underneath the harness the sandbox refuses three requests for three different reasons — rate limit, engagement boundary, and cloud metadata — without consulting the model at all.

## Your turn

Write your engagement scope as a data structure your harness reads, not as a paragraph in a PDF. Then ask what your current tooling would do if a target redirected to a host you were not authorised to touch.

---

**Next → [C1.2 · Red-teaming an agent: designing the campaign](https://spbreed.github.io/cyber-commons/lessons/C1.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C1.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C1.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*